# AI101 – Assignment 5a: Chicago Crime Data Analysis with CrewAI Agents

This notebook uses the **Chicago Crime dataset** from Kaggle (selected in Assignment 1) and replicates the multi-agent CrewAI pipeline from Assignment 5.

Two agents work in sequence:
1. A **Crime Data Analyst** reads and summarizes patterns in the data
2. A **Crime Trend Forecaster** uses that analysis to predict future trends and recommend resource allocation

**Prompts used throughout this assignment are documented in the final cell.**

In [31]:
# Mount Google Drive so Colab can access files stored there.
# force_remount=True ensures a fresh connection each session.
# userdata is imported in case we later pull the API key from Colab Secrets.
from google.colab import userdata, drive
drive.mount("/content/drive", force_remount=True)

Mounted at /content/drive


In [20]:
import os
from crewai import Agent, Task, Crew, Process, LLM
import google.generativeai as genai  # Needed to configure the Gemini API key
from crewai.tools import BaseTool
import json # Added to check file type
import pandas as pd # Added for robust error handling, though primarily used later
from google.colab import userdata, drive
drive.mount("/content/drive", force_remount=True)

# Replace with your actual Gemini API key
# gemini_api_key = "xxxxxxxxxxxxxxxxxxxx"
gemini_api_key = userdata.get('GOOGLE_API_KEY') # Securely get API key from Colab secrets

# Configure the Gemini API so the google.generativeai library can authenticate
genai.configure(api_key=gemini_api_key)

# Path to the Chicago Crime CSV in Google Drive
# Update this if your file is stored in a different folder
file_path = '/content/drive/My Drive/Colab Notebooks/chicago_crime.csv'

# We read the CSV as a plain string (not a DataFrame) because we need to
# pass the raw data directly into the agent's prompt as text.
# The agent reads it like a document, interpreting the structure itself.
try:
    with open(file_path, 'r') as f:
        file_content_raw = f.read()

    # Check if the file content looks like a Jupyter Notebook (JSON with 'cells' key)
    is_notebook_file = False
    try:
        parsed_json = json.loads(file_content_raw)
        if isinstance(parsed_json, dict) and "cells" in parsed_json and isinstance(parsed_json["cells"], list):
            is_notebook_file = True
    except json.JSONDecodeError:
        pass # Not a valid JSON, so definitely not a notebook.

    if is_notebook_file:
        print("WARNING: The file at specified path ('chicago_crime.csv') appears to be a Jupyter Notebook (.ipynb) file, not a CSV.")
        print("Please ensure this file contains actual Chicago crime data in CSV format.")
        print("For now, a dummy CSV string will be used to allow the notebook to proceed, but analysis will be on dummy data.")
        csv_string = "ID,Case Number,Date,Block,IUCR,Primary Type,Description,Location Description,Arrest,Domestic,Beat,District,Ward,Community Area,FBI Code,X Coordinate,Y Coordinate,Year,Updated On,Latitude,Longitude,Location\n" \
                     "1,HZ123456,01/01/2023 12:00:00 AM,0100XX S STATE ST,1150,DECEPTIVE PRACTICE,UNLAWFUL USE OF A CREDIT CARD,STREET,FALSE,FALSE,0111,001,42,32,11,1177691,1899999,2023,02/10/2023 03:00:00 PM,41.868725,-87.627763,{'latitude': '41.868725', 'longitude': '-87.627763', 'location': '(41.868725, -87.627763)'}\n" \
                     "2,HZ123457,01/01/2023 01:00:00 AM,0200XX W RANDOLPH ST,0460,BATTERY,SIMPLE,APARTMENT,TRUE,FALSE,0112,001,42,32,04B,1177692,1900001,2023,02/10/2023 03:00:00 PM,41.884841,-87.632194,{'latitude': '41.884841', 'longitude': '-87.632194', 'location': '(41.884841, -87.632194)'}"
    else:
        # Assume it's a CSV if not a detected notebook
        csv_string = file_content_raw
        print("File content read successfully!")
        print(csv_string[:500]) # Preview the first 500 characters to verify the file loaded correctly

except FileNotFoundError:
    print(f"File not found at {file_path}. Please check the path and try again.")
    print("For now, a dummy CSV string will be used to allow the notebook to proceed, but analysis will be on dummy data.")
    # Provide a dummy CSV string if file is not found
    csv_string = "ID,Case Number,Date,Block,IUCR,Primary Type,Description,Location Description,Arrest,Domestic,Beat,District,Ward,Community Area,FBI Code,X Coordinate,Y Coordinate,Year,Updated On,Latitude,Longitude,Location\n" \
                 "1,HZ123456,01/01/2023 12:00:00 AM,0100XX S STATE ST,1150,DECEPTIVE PRACTICE,UNLAWFUL USE OF A CREDIT CARD,STREET,FALSE,FALSE,0111,001,42,32,11,1177691,1899999,2023,02/10/2023 03:00:00 PM,41.868725,-87.627763,{'latitude': '41.868725', 'longitude': '-87.627763', 'location': '(41.868725, -87.627763)'}\n" \
                 "2,HZ123457,01/01/2023 01:00:00 AM,0200XX W RANDOLPH ST,0460,BATTERY,SIMPLE,APARTMENT,TRUE,FALSE,0112,001,42,32,04B,1177692,1900001,2023,02/10/2023 03:00:00 PM,41.884841,-87.632194,{'latitude': '41.884841', 'longitude': '-87.632194', 'location': '(41.884841, -87.632194)'}"
except Exception as e:
    print(f"An unexpected error occurred while reading the file: {e}")
    print("For now, a dummy CSV string will be used to allow the notebook to proceed, but analysis will be on dummy data.")
    # Provide a dummy CSV string if any other error occurs
    csv_string = "ID,Case Number,Date,Block,IUCR,Primary Type,Description,Location Description,Arrest,Domestic,Beat,District,Ward,Community Area,FBI Code,X Coordinate,Y Coordinate,Year,Updated On,Latitude,Longitude,Location\n" \
                 "1,HZ123456,01/01/2023 12:00:00 AM,0100XX S STATE ST,1150,DECEPTIVE PRACTICE,UNLAWFUL USE OF A CREDIT CARD,STREET,FALSE,FALSE,0111,001,42,32,11,1177691,1899999,2023,02/10/2023 03:00:00 PM,41.868725,-87.627763,{'latitude': '41.868725', 'longitude': '-87.627763', 'location': '(41.868725, -87.627763)'}\n" \
                 "2,HZ123457,01/01/2023 01:00:00 AM,0200XX W RANDOLPH ST,0460,BATTERY,SIMPLE,APARTMENT,TRUE,FALSE,0112,001,42,32,04B,1177692,1900001,2023,02/10/2023 03:00:00 PM,41.884841,-87.632194,{'latitude': '41.884841', 'longitude': '-87.632194', 'location': '(41.884841, -87.632194)'}"

Mounted at /content/drive
Please ensure this file contains actual Chicago crime data in CSV format.
For now, a dummy CSV string will be used to allow the notebook to proceed, but analysis will be on dummy data.


In [21]:
# Initialize the Gemini LLM that will power both agents.
# model: we use gemini-2.0-flash-lite as a stable, rate-limit-friendly option.
#        Switch to gemini/gemini-2.5-pro for more powerful analysis if not rate-limited.
# api_key: authenticates our requests to the Gemini API.
# temperature=0.0: low temperature means the model gives consistent, deterministic
#        outputs — less creative guessing, more grounded in the actual data.
gemini_llm = LLM(
    model='gemini/gemini-2.0-flash-lite',  ### Switch to 'gemini/gemini-2.5-pro' if not rate-limited
    api_key=gemini_api_key,
    temperature=0.0
)

In [22]:
# Quick sanity check — print the first 500 characters of the CSV string.
# This confirms the file was read correctly and shows us the column headers
# and a few rows before passing data to the agents.
print(csv_string[:500])

ID,Case Number,Date,Block,IUCR,Primary Type,Description,Location Description,Arrest,Domestic,Beat,District,Ward,Community Area,FBI Code,X Coordinate,Y Coordinate,Year,Updated On,Latitude,Longitude,Location
1,HZ123456,01/01/2023 12:00:00 AM,0100XX S STATE ST,1150,DECEPTIVE PRACTICE,UNLAWFUL USE OF A CREDIT CARD,STREET,FALSE,FALSE,0111,001,42,32,11,1177691,1899999,2023,02/10/2023 03:00:00 PM,41.868725,-87.627763,{'latitude': '41.868725', 'longitude': '-87.627763', 'location': '(41.868725, -87.6277


In [23]:
# The Chicago Crime dataset can be very large (millions of rows).
# Passing the full CSV to an LLM would exceed its token limit and cause errors.
# We sample 500 rows to keep the prompt manageable while still representing
# the data's structure, column names, and value distributions.
import pandas as pd
import io

# Parse the full CSV string into a DataFrame so we can sample it
df_crime = pd.read_csv(io.StringIO(csv_string))
print(f"Full dataset: {df_crime.shape[0]:,} rows x {df_crime.shape[1]} columns")
print("Columns:", df_crime.columns.tolist())

# Sample 500 rows with a fixed random_state for reproducibility
df_sample = df_crime.sample(n=min(500, len(df_crime)), random_state=42)

# Convert the sample back into a CSV string to pass to the agent
csv_string_sample = df_sample.to_csv(index=False)
print(f"\nSample size passed to agents: {len(df_sample)} rows")

Full dataset: 2 rows x 22 columns
Columns: ['ID', 'Case Number', 'Date', 'Block', 'IUCR', 'Primary Type', 'Description', 'Location Description', 'Arrest', 'Domestic', 'Beat', 'District', 'Ward', 'Community Area', 'FBI Code', 'X Coordinate', 'Y Coordinate', 'Year', 'Updated On', 'Latitude', 'Longitude', 'Location']

Sample size passed to agents: 2 rows


In [24]:
from crewai import Agent, Task, Crew, Process
from crewai.tools import BaseTool
import pandas as pd

# CsvStringReaderTool is a custom CrewAI tool that wraps our CSV data.
# Agents can call this tool to retrieve the data instead of having it
# baked directly into the task description — this is a cleaner pattern
# when data is large or needs to be reused across multiple agents.
#
# BaseTool requires:
#   name        – how the agent refers to this tool
#   description – tells the agent what the tool does so it knows when to use it
#   _run()      – the method that actually executes when the tool is called
class CsvStringReaderTool(BaseTool):
    name: str = "CSV String Reader"
    description: str = "Reads the Chicago crime data from the provided CSV string and returns it for analysis."
    csv_data: str

    def _run(self, query: str = None) -> str:
        """Returns the full CSV data string when called by an agent."""
        return self.csv_data

# Instantiate the tool with our 500-row sample CSV string
csv_reader_tool = CsvStringReaderTool(csv_data=csv_string_sample)

# Agent 1: Crime Data Analyst
# This agent's job is to read the raw Chicago crime data and identify patterns.
# role      – the agent's job title; shapes how it frames its output
# goal      – what the agent is trying to accomplish in this task
# backstory – gives the agent context about its expertise so it reasons appropriately
# verbose   – prints the agent's internal reasoning steps so we can follow along
# allow_delegation=False – this agent works independently, does not hand off to others
# tools     – the CSV reader tool gives the agent access to our dataset
data_analyst = Agent(
    role='Crime Data Analyst',
    goal='Analyze the Chicago crime dataset to identify the most common crime types, hotspot districts, and time-based trends.',
    backstory=(
        'You are a data analyst specializing in urban public safety statistics. '
        'You have years of experience working with city crime databases and can quickly '
        'identify which crime categories, neighborhoods, and time periods show the most activity. '
        'You present your findings clearly with supporting evidence from the data.'
    ),
    verbose=True,
    allow_delegation=False,
    llm=gemini_llm,
    tools=[csv_reader_tool]  # Give this agent access to the crime data
)

# Task for Agent 1: Analyze the crime data
# description – the prompt the agent receives; we reference the CSV tool so the agent knows to call it
# expected_output – tells the agent what a complete, correct response looks like
# agent – assigns this task to the crime data analyst
#
# Prompt used: Analyze crime types, districts, time trends, and arrest rates
analyze_data_task = Task(
    description=f"""
Use the CSV String Reader tool to retrieve the Chicago crime dataset, then analyze it thoroughly.

Your analysis should cover:
1. The most common crime types and how frequently they appear
2. Which districts or community areas have the highest crime counts
3. Time-based trends — crime patterns by year, month, day of week, or hour (use whatever columns are available)
4. The proportion of crimes that resulted in an arrest
5. Any notable patterns, anomalies, or correlations in the data

Be specific and reference actual values from the data in your report.
""",
    expected_output='A structured analytical report summarizing crime type frequencies, geographic hotspots, time-based trends, arrest rates, and notable findings from the Chicago crime dataset.',
    agent=data_analyst
)

In [30]:
# Agent 2: Crime Trend Forecaster
# This agent takes Agent 1's analysis as context and produces forward-looking predictions.
# It does NOT have the csv_reader_tool because it doesn't need raw data —
# it works entirely from the structured report that Agent 1 produces.
crime_forecaster = Agent(
    role='Crime Trend Forecaster',
    goal='Predict future crime trends in Chicago and recommend where law enforcement resources should be prioritized.',
    backstory=(
        'You are a public safety strategist with expertise in predictive policing and urban crime forecasting. '
        'You use historical crime patterns identified by analysts to make forward-looking predictions '
        'and generate actionable recommendations for law enforcement agencies and city planners.'
    ),
    verbose=True,
    allow_delegation=False,
    llm=gemini_llm
)

# Print first 500 chars as a final confirmation the data is ready before running the crew
print(csv_string_sample[0:500])

# Task for Agent 2: Forecast crime trends based on Agent 1's report
# context=[analyze_data_task] is the key line — it passes Agent 1's output
# as input to Agent 2, creating the hierarchical pipeline.
# Agent 2 cannot start until Agent 1 finishes.
#
# Prompt used: Predict 6-month trends, identify high-risk districts, recommend resource allocation
predict_crime_trends_task = Task(
    description=f"""
Based on the crime analysis report provided by the Crime Data Analyst, forecast future crime trends for Chicago.

Your forecast should address:
1. Which crime types are likely to increase or decrease over the next 6 months, and why
2. Which districts should receive the most law enforcement attention based on historical patterns
3. Whether there are seasonal or cyclical patterns that should inform resource planning
4. Concrete, actionable recommendations for law enforcement and city officials

Ground every prediction in the findings from the data analysis report.
""",
    expected_output='A forward-looking crime trend forecast with specific predictions by crime type and district, seasonal planning insights, and concrete resource allocation recommendations for Chicago law enforcement.',
    agent=crime_forecaster,
    context=[analyze_data_task]  # Agent 1's analysis is passed as context to Agent 2
)

# Assemble the Chicago Crime Crew
# agents – both agents listed in execution order
# tasks  – both tasks listed in execution order
# process=Process.hierarchical – a manager LLM coordinates the crew;
#          each task's output flows into the next as context
# manager_llm – the model that acts as the crew manager and delegates work
chicago_crime_crew = Crew(
    agents=[data_analyst, crime_forecaster],
    tasks=[analyze_data_task, predict_crime_trends_task],
    process=Process.hierarchical,
    manager_llm=gemini_llm
)

# Kick off the crew — Agent 1 analyzes first, then Agent 2 forecasts
print("## Starting the Chicago Crime Analysis Crew")
try:
    result = chicago_crime_crew.kickoff()
    # Print the combined final report from both agents
    print("## Final Report:")
    print(result)
except Exception as e:
    print("\n## Execution Error ##")
    print(f"{e}")
    if "429" in str(e):
        print("\n*** QUOTA EXCEEDED (429) ***")
        print("You have hit the Gemini API rate limit for your tier.")
        print("Please wait a few minutes before trying again, or check your API billing details.")


ID,Case Number,Date,Block,IUCR,Primary Type,Description,Location Description,Arrest,Domestic,Beat,District,Ward,Community Area,FBI Code,X Coordinate,Y Coordinate,Year,Updated On,Latitude,Longitude,Location
0200XX W RANDOLPH ST,460,BATTERY,SIMPLE,APARTMENT,True,False,112,1,42,32,04B,1177692,1900001,2023,02/10/2023 03:00:00 PM,41.884841,-87.632194,{'latitude': '41.884841', 'longitude': '-87.632194', 'location': '(41.884841, -87.632194)'}
0100XX S STATE ST,1150,DECEPTIVE PRACTICE,UNLAWFUL USE OF A 
## Starting the Chicago Crime Analysis Crew


ERROR:root:Google Gemini API error: 429 - You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. 
* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 0, model: gemini-2.0-flash-lite
* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 0, model: gemini-2.0-flash-lite
* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_input_token_count, limit: 0, model: gemini-2.0-flash-lite
Please retry in 36.926055279s.
ERROR:root:Google Gemini API error: 429 - You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https

[CrewAIEventsBus] Warning: Event pairing mismatch. 'task_failed' closed 'agent_execution_started' (expected 
'task_started')

[CrewAIEventsBus] Warning: Event pairing mismatch. 'crew_kickoff_failed' closed 'agent_execution_started' (expected
'crew_kickoff_started')


## Execution Error ##
429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 0, model: gemini-2.0-flash-lite\n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 0, model: gemini-2.0-flash-lite\n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_input_token_count, limit: 0, model: gemini-2.0-flash-lite\nPlease retry in 36.651444334s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai.google.dev/gemini-api/docs/rate-limit

╭─────────────────────────────────────────── Tracing Preference Saved ────────────────────────────────────────────╮
│                                                                                                                 │
│  Info: Tracing has been disabled.                                                                               │
│                                                                                                                 │
│  Your preference has been saved. Future Crew/Flow executions will not collect traces.                           │
│                                                                                                                 │
│  To enable tracing later, do any one of these:                                                                  │
│  • Set tracing=True in your Crew/Flow code                                                                      │
│  • Set CREWAI_TRACING_ENABLED=true in your project's .env file                                                  │
│  • Run: crewai traces enable                                                                                    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

In [27]:
file_path_to_check = '/content/drive/My Drive/Colab Notebooks/chicago_crime.csv'

try:
    with open(file_path_to_check, 'r') as f:
        first_lines = [next(f) for _ in range(5)] # Read first 5 lines
    print("--- First 5 lines of the file ---")
    for line in first_lines:
        print(line.strip())
    print("-----------------------------------")
except FileNotFoundError:
    print(f"File not found at {file_path_to_check}.")
except Exception as e:
    print(f"An error occurred while reading the file: {e}")


--- First 5 lines of the file ---
{
"cells": [
{
"cell_type": "markdown",
"metadata": {},
-----------------------------------


In [26]:
# Example: If you have your data in a DataFrame called `my_crime_df`
# Make sure to replace `my_crime_df` with the actual name of your DataFrame
# and verify the path.

# Assuming 'my_crime_df' is your pandas DataFrame
# my_crime_df.to_csv('/content/drive/My Drive/Colab Notebooks/chicago_crime.csv', index=False)
# print("Successfully saved DataFrame to chicago_crime.csv")

# Uncomment the lines above and run this cell if you need to save a DataFrame as a CSV.

Once you've confirmed and corrected the `chicago_crime.csv` file to contain proper CSV data, you should re-run the cells from `HJNoRBkI1TY6` onwards to ensure the agents load the correct data.

## Output Discussion & Prompt Documentation

### What the output represents

The crew produces a two-part report:

**Part 1 – Crime Analysis Report (Agent 1: Crime Data Analyst)**
A descriptive summary of the Chicago crime dataset sample — the most frequent crime types, which districts have the highest incident counts, time-based trends (by year, month, or hour), arrest rates, and any notable anomalies. This agent uses the `CsvStringReaderTool` to directly access the data rather than relying on what was baked into the task description.

**Part 2 – Crime Trend Forecast (Agent 2: Crime Trend Forecaster)**
A forward-looking prediction report built on Agent 1's findings. This includes which crime categories are likely to rise or fall over the next 6 months, which districts need prioritized resources, seasonal planning insights, and concrete recommendations for law enforcement. Agent 2 never sees the raw data — it reasons entirely from Agent 1's structured report, passed in via `context=[analyze_data_task]`.

The `verbose=True` setting on both agents prints their internal reasoning steps (chain of thought) so you can follow how each agent works through the problem before producing its final answer.

### Prompts used throughout this assignment

1. **Agent 1 task description prompt:** *"Use the CSV String Reader tool to retrieve the Chicago crime dataset, then analyze it. Cover: most common crime types and frequencies, districts with highest crime counts, time-based trends, arrest rates, and notable patterns or anomalies."*

2. **Agent 2 task description prompt:** *"Based on the crime analysis report provided, forecast future crime trends. Address: which crime types are likely to increase or decrease over the next 6 months, which districts need the most attention, seasonal/cyclical patterns, and concrete resource allocation recommendations."*

3. **Sampling rationale prompt (used to design the preprocessing step):** *"The Chicago Crime dataset has millions of rows — sample 500 rows for the agent prompt to stay within token limits while preserving the data's structure and column diversity."*